Section 1 — Load & Basic Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../artifacts/raw.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

Dataset has 7043 rows and 21 columns.
TotalCharges is stored as object (string) — needs to be converted to float.
No missing values on surface, but TotalCharges has blank spaces hiding as strings.
customerID is just an identifier — will be dropped before training.

Section 2 — Target Distribution (critical for imbalance check)

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['Churn'].value_counts(normalize=True) * 100

In [ ]:
sns.countplot(x='Churn', data=df)

26.5% customers churned (Yes), 73.5% did not (No).
This is imbalanced — a model can get 73% accuracy by predicting "No" every time, which is useless.
We will use SMOTE in data transformation to balance this out.

Section 3 — Fix known data issue

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isnull().sum()

In [ ]:
df[df['TotalCharges'].isnull()] 

11 rows had blank spaces in TotalCharges — these became NaN after conversion.
All 11 of these customers have tenure = 0, meaning they just signed up and were never billed.
Will handle these NaNs with median imputation in the pipeline.

Section 4 — Numerical features vs Churn

In [ ]:
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    sns.boxplot(x='Churn', y=col, data=df)
    plt.show()

Customers who churned tend to have lower tenure — they leave early.
Monthly charges are higher for churned customers.
TotalCharges is lower for churned customers because they didn't stay long enough to accumulate charges.
Tenure is the strongest numerical signal for churn.

Section 5 — Categorical features vs Churn

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender', 'Partner', 'Dependents']

for col in cat_cols:
    sns.countplot(x=col, hue='Churn', data=df)
    plt.xticks(rotation=30)
    plt.show()

Month-to-month contract customers churn the most — no lock-in means easy to leave.
Customers with Fiber Optic internet churn more than DSL — possibly due to higher cost.
Customers without OnlineSecurity, TechSupport, or OnlineBackup churn more.
Electronic check payment method has the highest churn rate.
Gender has almost no effect on churn — nearly equal split.

Section 6 — Correlation heatmap (numerics only)

In [ ]:
sns.heatmap(df[['tenure','MonthlyCharges','TotalCharges']].corr(), annot=True, cmap='coolwarm')

Tenure and TotalCharges are highly correlated (0.83) — longer you stay, more you pay total.
MonthlyCharges and TotalCharges are moderately correlated (0.65).
No multicollinearity issue severe enough to drop columns — the pipeline will handle scaling.

## What we learned from EDA

Features that matter most: Contract type, tenure, MonthlyCharges, InternetService, OnlineSecurity
Data issues to fix: TotalCharges type conversion, 11 NaN values, drop customerID
Target is imbalanced (74/26) — need SMOTE
All categorical columns need encoding, all numerical columns need scaling
SeniorCitizen is already 0/1 — treat as numerical